In [4]:
!pip install -q "gradio>=5,<6" groq

In [ ]:
# =====================  NutriStudio — Groq + Gradio  =====================
# In Colab, run this FIRST in its own cell:
#     !pip install -q "gradio>=5,<6" groq
# Then add your key under the 🔑 Secrets tab as GROQ_API_KEY (turn on notebook access),
# and run this whole file in a second cell.
# =========================================================================

import os, re, json, html
from datetime import datetime
from getpass import getpass

import pandas as pd
import gradio as gr
from groq import Groq

# ---------------------------- API key & model ----------------------------
def get_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")          # "GROQ_API_KEY" = NAME of your secret
        if k:
            return k
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY") or getpass("Paste your Groq API key: ")

client = Groq(api_key=get_key())
MODEL = "openai/gpt-oss-120b"        # lighter alternative: "openai/gpt-oss-20b"

# ------------------------------ shared state -----------------------------
DIARY = []                                            # list of food entries for today
TARGETS = {"kcal": 2000, "protein": 100, "carbs": 250, "fat": 65}


def num(x):
    try:
        return float(x)
    except Exception:
        return 0.0


def totals():
    return {k: round(sum(num(e[k]) for e in DIARY), 1) for k in ("kcal", "protein", "carbs", "fat")}


def llm_json(system, user, temperature=0.2):
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"},
    )
    txt = (r.choices[0].message.content or "{}").strip()
    txt = re.sub(r"^```(?:json)?|```$", "", txt, flags=re.M).strip()
    return json.loads(txt)


def compute_targets(age, sex, height_cm, weight_kg, activity="moderate", goal="maintain"):
    """Mifflin-St Jeor -> TDEE -> calorie & macro targets (also stored in TARGETS)."""
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + (5 if str(sex).lower().startswith("m") else -161)
    factor = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9}
    tdee = bmr * factor.get(activity, 1.55)
    kcal = max(tdee + {"lose": -500, "maintain": 0, "gain": 400}.get(goal, 0), 1200)
    protein = weight_kg * (1.4 if goal == "maintain" else 1.8)
    fat = kcal * 0.25 / 9
    carbs = (kcal - protein * 4 - fat * 9) / 4
    TARGETS.update({"kcal": round(kcal), "protein": round(protein), "carbs": round(carbs), "fat": round(fat)})
    return {"bmr": round(bmr), "tdee": round(tdee), **TARGETS}


# =============================== TAB 1: ANALYZER ==========================
ANALYZE_SYS = """You are an expert nutritionist. Reply with ONLY a JSON object:
{"food": str, "edible": bool, "verdict": "SAFE TO EAT"|"EAT IN MODERATION"|"AVOID",
 "health_score": number 1-10, "reason": "1-2 sentences", "serving": str,
 "calories_kcal": number, "protein_g": number, "carbs_g": number, "fat_g": number,
 "fiber_g": number, "sugar_g": number, "sodium_mg": number,
 "vitamins_minerals": [3-5 str], "benefits": [1-3 str], "warnings": [str], "tip": str,
 "healthier_swap": str}
Rules: if the input is not food or is toxic (soap, plastic, poisonous mushroom...), set edible=false,
verdict="AVOID", health_score=0, numbers 0, and explain why. If no serving is given, assume a typical
single serving and state it. Respect the user's health conditions/goals and allergies in the verdict
and warnings (an allergen match must be AVOID). Values are approximate estimates."""

COLORS = {"SAFE TO EAT": "#16a34a", "EAT IN MODERATION": "#d97706", "AVOID": "#dc2626"}
ICONS = {"SAFE TO EAT": "✅", "EAT IN MODERATION": "⚠️", "AVOID": "❌"}
esc = lambda s: html.escape(str(s))


def verdict_card(d):
    v = d.get("verdict", "EAT IN MODERATION")
    c = COLORS.get(v, "#64748b")
    score = max(0, min(10, num(d.get("health_score", 0))))
    chips = "".join(
        f'<span style="display:inline-block;margin:3px 4px 0 0;padding:3px 10px;border-radius:999px;'
        f'background:rgba(220,38,38,.15);color:#dc2626;font-size:13px">⚠ {esc(w)}</span>'
        for w in d.get("warnings", []) or [])
    return f"""
<div style="border:2px solid {c};border-radius:16px;padding:18px;background:{c}14">
  <div style="font-size:22px;font-weight:700;color:{c}">{ICONS.get(v, 'ℹ️')} {esc(v)}</div>
  <div style="opacity:.8;margin:2px 0 8px">{esc(d.get('food', ''))} · {esc(d.get('serving', ''))}</div>
  <div style="font-size:42px;font-weight:800;line-height:1">{num(d.get('calories_kcal')):g}
     <span style="font-size:16px;font-weight:500">kcal</span></div>
  <div style="margin:10px 0 4px;font-size:13px">Health score: <b>{score:g}/10</b></div>
  <div style="height:10px;border-radius:8px;background:rgba(128,128,128,.25);overflow:hidden">
     <div style="width:{score * 10}%;height:100%;background:{c}"></div></div>
  <p style="margin:12px 0 4px">{esc(d.get('reason', ''))}</p>
  <div>{chips}</div>
</div>"""


def analyze(food, qty, unit, conditions, allergies, other):
    if not (food or "").strip():
        raise gr.Error("Please enter a food name.")
    al = list(allergies or []) + ([other.strip()] if other and other.strip() else [])
    user = (f"Food: {food.strip()}\nServing: {qty:g} {unit}\n"
            f"Health conditions/goals: {', '.join(conditions or []) or 'none'}\n"
            f"Allergies: {', '.join(al) or 'none'}")
    try:
        d = llm_json(ANALYZE_SYS, user)
    except Exception as e:
        raise gr.Error(f"Groq error: {e}")

    plot = pd.DataFrame({"Nutrient": ["Protein", "Carbs", "Fat", "Fiber", "Sugar"],
                         "Grams": [num(d.get(k)) for k in
                                   ("protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g")]})
    bl = lambda items: "\n".join(f"- {i}" for i in items) if items else "- None"
    details = (f"**🧂 Sodium:** {num(d.get('sodium_mg')):g} mg\n\n"
               f"**🧬 Key nutrients**\n{bl(d.get('vitamins_minerals'))}\n\n"
               f"**🌿 Benefits**\n{bl(d.get('benefits'))}\n\n"
               f"**💡 Tip:** {d.get('tip', '')}\n\n"
               f"**🔁 Healthier swap:** {d.get('healthier_swap', '—')}\n\n"
               "<sub>Estimates only — not medical advice.</sub>")
    return verdict_card(d), plot, details, d


def add_to_diary(last, meal):
    if not last or not last.get("edible", False):
        gr.Warning("Analyze an edible food first.")
        return
    DIARY.append({"time": datetime.now().strftime("%H:%M"), "meal": meal,
                  "food": last.get("food", "?"), "serving": last.get("serving", ""),
                  "kcal": num(last.get("calories_kcal")), "protein": num(last.get("protein_g")),
                  "carbs": num(last.get("carbs_g")), "fat": num(last.get("fat_g"))})
    gr.Info(f"Added {last.get('food', 'food')} to {meal} ✔")


# =============================== TAB 2: CHAT AGENT ========================
def t_log_food(food, meal="snack", kcal=0, protein_g=0, carbs_g=0, fat_g=0, serving="1 serving"):
    DIARY.append({"time": datetime.now().strftime("%H:%M"), "meal": meal, "food": food,
                  "serving": serving, "kcal": num(kcal), "protein": num(protein_g),
                  "carbs": num(carbs_g), "fat": num(fat_g)})
    return {"logged": food, "today_totals": totals(), "targets": TARGETS}


def t_summary():
    return {"totals": totals(), "targets": TARGETS, "items_logged": len(DIARY)}


def t_targets(age, sex, height_cm, weight_kg, activity="moderate", goal="maintain"):
    return compute_targets(age, sex, height_cm, weight_kg, activity, goal)


AGENT_TOOLS = {"log_food": t_log_food, "get_today_summary": t_summary, "calculate_targets": t_targets}
TOOL_SPECS = [
    {"type": "function", "function": {
        "name": "log_food",
        "description": "Log something the user ate. Estimate kcal and macros yourself for the portion.",
        "parameters": {"type": "object", "properties": {
            "food": {"type": "string"},
            "meal": {"type": "string", "enum": ["breakfast", "lunch", "dinner", "snack"]},
            "kcal": {"type": "number"}, "protein_g": {"type": "number"},
            "carbs_g": {"type": "number"}, "fat_g": {"type": "number"},
            "serving": {"type": "string"}}, "required": ["food", "meal", "kcal"]}}},
    {"type": "function", "function": {
        "name": "get_today_summary",
        "description": "Today's eaten totals versus the daily targets.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "calculate_targets",
        "description": "Compute and save daily calorie/macro targets from body stats and goal.",
        "parameters": {"type": "object", "properties": {
            "age": {"type": "integer"}, "sex": {"type": "string", "enum": ["male", "female"]},
            "height_cm": {"type": "number"}, "weight_kg": {"type": "number"},
            "activity": {"type": "string", "enum": ["sedentary", "light", "moderate", "active", "very_active"]},
            "goal": {"type": "string", "enum": ["lose", "maintain", "gain"]}},
            "required": ["age", "sex", "height_cm", "weight_kg"]}}},
]
AGENT_SYS = ("You are NutriBot, a friendly, concise nutrition coach. When the user says they ate "
             "something, estimate the nutrition and call log_food. When they share body stats, call "
             "calculate_targets. When they ask how they're doing, call get_today_summary. Never invent "
             "tool results. Suggest what to eat next based on remaining calories/macros. For medical "
             "conditions, pregnancy or eating disorders, recommend a qualified professional.")


def run_agent(msgs, max_steps=6):
    used = []
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOL_SPECS,
                                           tool_choice="auto", temperature=0.3)
        m = r.choices[0].message
        if not m.tool_calls:
            foot = f"\n\n<sub>🔧 Used: {', '.join(used)}</sub>" if used else ""
            return (m.content or "") + foot
        msgs.append({"role": "assistant", "content": m.content or "",
                     "tool_calls": [{"id": t.id, "type": "function",
                                     "function": {"name": t.function.name,
                                                  "arguments": t.function.arguments}}
                                    for t in m.tool_calls]})
        for t in m.tool_calls:
            try:
                res = AGENT_TOOLS[t.function.name](**json.loads(t.function.arguments or "{}"))
            except Exception as e:
                res = {"error": str(e)}
            used.append(t.function.name)
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(res)})
    return "Sorry, I couldn't finish that. Could you rephrase?"


# =============================== TAB 3: DASHBOARD =========================
def pbar(label, val, target, color):
    pct = round(100 * val / target) if target else 0
    warn = " ⚠️" if pct > 100 else ""
    return (f'<div style="margin:10px 0"><div style="display:flex;justify-content:space-between;'
            f'font-size:14px"><b>{label}</b><span>{val:g} / {target:g} ({pct}%){warn}</span></div>'
            f'<div style="height:12px;border-radius:8px;background:rgba(128,128,128,.25);overflow:hidden">'
            f'<div style="width:{min(pct, 100)}%;height:100%;background:{color}"></div></div></div>')


def dash():
    t = totals()
    left = max(TARGETS["kcal"] - t["kcal"], 0)
    head = (f'<div style="font-size:15px;margin-bottom:6px">🔥 <b>{left:g} kcal</b> remaining today</div>')
    bars = (head + pbar("Calories", t["kcal"], TARGETS["kcal"], "#16a34a")
            + pbar("Protein (g)", t["protein"], TARGETS["protein"], "#2563eb")
            + pbar("Carbs (g)", t["carbs"], TARGETS["carbs"], "#d97706")
            + pbar("Fat (g)", t["fat"], TARGETS["fat"], "#dc2626"))
    cols = ["time", "meal", "food", "serving", "kcal", "protein", "carbs", "fat"]
    df = pd.DataFrame(DIARY, columns=cols) if DIARY else pd.DataFrame(columns=cols)
    plot = pd.DataFrame({"Macro": ["Protein", "Carbs", "Fat"],
                         "Grams": [t["protein"], t["carbs"], t["fat"]]})
    return bars, df, plot


def set_targets_ui(age, sex, height, weight, activity, goal):
    r = compute_targets(int(age), sex, height, weight, activity, goal)
    msg = (f"**BMR** {r['bmr']} · **TDEE** {r['tdee']} kcal  \n"
           f"🎯 **{r['kcal']} kcal** · P {r['protein']} g · C {r['carbs']} g · F {r['fat']} g")
    return (msg, *dash())


def clear_diary():
    DIARY.clear()
    gr.Info("Diary cleared")
    return dash()


# =============================== TAB 4: MEAL PLANNER ======================
PLAN_SYS = """You are a dietitian. Reply with ONLY JSON:
{"total_kcal": number, "meals": [{"name": str, "items": [{"food": str, "qty": str, "kcal": number}],
 "kcal": number, "protein_g": number}], "tips": [str]}
Create a practical one-day meal plan that hits the calorie target (±5%), follows the diet type and
cuisine, avoids the listed foods, and fits the goal."""


def make_plan(kcal, diet, cuisine, n_meals, avoid, goal):
    user = (f"Calories: {kcal:g}\nDiet: {diet}\nCuisine: {cuisine}\nMeals per day: {int(n_meals)}\n"
            f"Avoid: {avoid or 'nothing'}\nGoal: {goal}")
    try:
        d = llm_json(PLAN_SYS, user, temperature=0.6)
    except Exception as e:
        raise gr.Error(f"Groq error: {e}")
    out = [f"## 🍽 Your plan — about {num(d.get('total_kcal')):g} kcal\n"]
    for m in d.get("meals", []):
        out.append(f"### {m.get('name', 'Meal')} · {num(m.get('kcal')):g} kcal · "
                   f"{num(m.get('protein_g')):g} g protein\n| Food | Qty | kcal |\n|---|---|---|")
        for it in m.get("items", []):
            out.append(f"| {it.get('food', '')} | {it.get('qty', '')} | {num(it.get('kcal')):g} |")
        out.append("")
    if d.get("tips"):
        out.append("### 💡 Tips\n" + "\n".join(f"- {t}" for t in d["tips"]))
    return "\n".join(out)


# ================================== UI ====================================
CONDITIONS = ["Diabetes", "High BP", "High cholesterol", "Heart disease", "Kidney disease",
              "PCOS", "Weight loss", "Muscle gain"]
ALLERGIES = ["Peanuts", "Tree nuts", "Dairy", "Gluten", "Eggs", "Soy", "Shellfish"]
UNITS = ["serving", "g", "ml", "piece", "cup", "bowl", "plate", "slice", "tbsp"]

GR6 = int(gr.__version__.split(".")[0]) >= 6      # Gradio 6 moved `theme` to launch()
THEME = gr.themes.Soft(primary_hue="emerald")

with gr.Blocks(title="NutriStudio", **({} if GR6 else {"theme": THEME})) as demo:
    gr.Markdown("# 🥗 NutriStudio\nAnalyze foods, chat with an AI coach, track your day and plan meals.")

    last = gr.State({})

    with gr.Tabs():
        # ---- Analyzer
        with gr.Tab("🔍 Analyze food"):
            with gr.Row():
                with gr.Column(scale=1):
                    food = gr.Textbox(label="Food", placeholder="e.g. chicken biryani, 2 idli, raw egg")
                    with gr.Row():
                        qty = gr.Number(label="Quantity", value=1, minimum=0.1)
                        unit = gr.Dropdown(UNITS, value="serving", label="Unit")
                    conditions = gr.CheckboxGroup(CONDITIONS, label="Health conditions / goals")
                    allergies = gr.CheckboxGroup(ALLERGIES, label="Allergies")
                    other = gr.Textbox(label="Other allergy (optional)")
                    go = gr.Button("Analyze 🔍", variant="primary")
                    gr.Examples([["Banana"], ["Chicken biryani"], ["Peanut butter toast"], ["Soap"]],
                                inputs=[food])
                with gr.Column(scale=1):
                    card = gr.HTML()
                    macro_plot = gr.BarPlot(
                        pd.DataFrame({"Nutrient": ["Protein", "Carbs", "Fat", "Fiber", "Sugar"],
                                      "Grams": [0, 0, 0, 0, 0]}),
                        x="Nutrient", y="Grams", title="Macros (g)", height=260)
                    details = gr.Markdown()
                    with gr.Row():
                        meal = gr.Dropdown(["breakfast", "lunch", "dinner", "snack"],
                                           value="lunch", label="Meal", scale=1)
                        add = gr.Button("➕ Add to today's diary", scale=2)

        # ---- Chat
        with gr.Tab("💬 AI coach") as chat_tab:
            try:                                           # Gradio 5
                chatbot = gr.Chatbot(type="messages", height=420, label="NutriBot")
            except TypeError:                              # Gradio 6 (messages is the default)
                chatbot = gr.Chatbot(height=420, label="NutriBot")
            with gr.Row():
                msg = gr.Textbox(placeholder="e.g. I had 2 dosa and sambar for breakfast",
                                 show_label=False, scale=5)
                send = gr.Button("Send", variant="primary", scale=1)
            gr.Examples(["I'm 28, male, 172 cm, 70 kg, moderately active, want to lose weight",
                         "I had 2 dosa with sambar for breakfast",
                         "How am I doing today?",
                         "What should I eat for dinner?"], inputs=[msg])

        # ---- Dashboard
        with gr.Tab("📒 Today") as dash_tab:
            with gr.Row():
                with gr.Column(scale=1):
                    bars = gr.HTML()
                    macro_chart = gr.BarPlot(x="Macro", y="Grams", title="Macros eaten today (g)",
                                             height=240)
                with gr.Column(scale=1):
                    with gr.Accordion("🎯 Set my daily targets", open=False):
                        with gr.Row():
                            age = gr.Slider(14, 90, value=28, step=1, label="Age")
                            sex = gr.Radio(["male", "female"], value="male", label="Sex")
                        with gr.Row():
                            height = gr.Slider(120, 210, value=170, step=1, label="Height (cm)")
                            weight = gr.Slider(35, 160, value=70, step=1, label="Weight (kg)")
                        activity = gr.Dropdown(["sedentary", "light", "moderate", "active", "very_active"],
                                               value="moderate", label="Activity")
                        goal = gr.Radio(["lose", "maintain", "gain"], value="maintain", label="Goal")
                        calc = gr.Button("Calculate targets")
                        calc_out = gr.Markdown()
            table = gr.Dataframe(label="Food diary", interactive=False)
            with gr.Row():
                refresh = gr.Button("🔄 Refresh")
                clear = gr.Button("🗑 Clear diary", variant="stop")

        # ---- Planner
        with gr.Tab("🍽 Meal planner"):
            with gr.Row():
                with gr.Column(scale=1):
                    p_kcal = gr.Slider(1000, 4000, value=2000, step=50, label="Calories / day")
                    p_diet = gr.Radio(["Vegetarian", "Non-vegetarian", "Vegan", "Eggetarian"],
                                      value="Vegetarian", label="Diet")
                    p_cuisine = gr.Dropdown(["South Indian", "North Indian", "Mediterranean",
                                             "Asian", "Western", "Any"], value="South Indian",
                                            label="Cuisine")
                    p_meals = gr.Slider(3, 6, value=4, step=1, label="Meals per day")
                    p_goal = gr.Radio(["Lose fat", "Maintain", "Build muscle"], value="Maintain",
                                      label="Goal")
                    p_avoid = gr.Textbox(label="Avoid (optional)", placeholder="e.g. mushrooms, fried food")
                    p_go = gr.Button("Generate plan ✨", variant="primary")
                with gr.Column(scale=1):
                    plan_out = gr.Markdown()

    # ---- wiring
    dash_outs = [bars, table, macro_chart]

    go.click(analyze, [food, qty, unit, conditions, allergies, other],
             [card, macro_plot, details, last])
    food.submit(analyze, [food, qty, unit, conditions, allergies, other],
                [card, macro_plot, details, last])
    add.click(add_to_diary, [last, meal], None).then(dash, None, dash_outs)

    def as_text(c):
        """Chat content may be a string or (Gradio 6) a list of content blocks."""
        if isinstance(c, str):
            return c
        if isinstance(c, dict):
            return str(c.get("text", ""))
        if isinstance(c, list):
            return " ".join(as_text(x) for x in c)
        return str(c or "")

    def chat(user_msg, history):
        if not (user_msg or "").strip():
            return "", history, *dash()
        history = list(history or []) + [{"role": "user", "content": user_msg}]
        msgs = [{"role": "system", "content": AGENT_SYS}] + [
            {"role": m["role"], "content": as_text(m["content"])} for m in history]
        try:
            reply = run_agent(msgs)
        except Exception as e:
            reply = f"⚠️ Error: {e}"
        history.append({"role": "assistant", "content": reply})
        return "", history, *dash()

    send.click(chat, [msg, chatbot], [msg, chatbot, *dash_outs])
    msg.submit(chat, [msg, chatbot], [msg, chatbot, *dash_outs])

    dash_tab.select(dash, None, dash_outs)
    refresh.click(dash, None, dash_outs)
    clear.click(clear_diary, None, dash_outs)
    calc.click(set_targets_ui, [age, sex, height, weight, activity, goal], [calc_out, *dash_outs])
    p_go.click(make_plan, [p_kcal, p_diet, p_cuisine, p_meals, p_avoid, p_goal], plan_out)
    demo.load(dash, None, dash_outs)

demo.launch(share=True, debug=True, **({"theme": THEME} if GR6 else {}))

Paste your Groq API key: ··········
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fa39d0ed6f2107e160.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
